# Human Gene Essentiality Predictor V2
## Hedge Fund + Expression Features for Context-Dependent Genes

**V1 Results:**
- Pan-essential (≥90%): 98.2% ✅
- Non-essential (0%): 100% ✅
- Context-dependent: 43.9% ❌ (worse than random!)

**V2 Goal:** Fix context-dependent prediction by adding expression features

## 1. Setup & Data Loading

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded!")

In [ ]:
# ============================================================
# DATA PATHS - Update to your Drive location
# ============================================================
DATA_DIR = '/content/drive/MyDrive/depmap_data'

# CRISPR file
CRISPR_PATH = os.path.join(DATA_DIR, 'CRISPRGeneEffect.csv')

# Expression file - try multiple possible names
EXPRESSION_NAMES = [
    'OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv',  # Your filename
    'OmicsExpressionProteinCodingGenesTPMLogp1.csv',       # Standard name
    'OmicsExpressionTPMLogp1HumanProteinCodingGenes',      # Without extension
]

EXPRESSION_PATH = None
for name in EXPRESSION_NAMES:
    path = os.path.join(DATA_DIR, name)
    if os.path.exists(path):
        EXPRESSION_PATH = path
        print(f"Found expression file: {name}")
        break

if EXPRESSION_PATH is None:
    print("Expression file NOT found!")
    print(f"Looking in: {DATA_DIR}")
    print(f"Files available: {os.listdir(DATA_DIR)}")

In [ ]:
# Load CRISPR essentiality data
print("Loading CRISPR data...")
crispr_df = pd.read_csv(CRISPR_PATH, index_col=0)
print(f"CRISPR shape: {crispr_df.shape} (cell_lines x genes)")

# Transpose: genes as rows, cell lines as columns
crispr_df = crispr_df.T
print(f"Transposed: {crispr_df.shape} (genes x cell_lines)")

In [ ]:
# Load Expression data
print("\nLoading Expression data...")
if EXPRESSION_PATH:
    expr_df = pd.read_csv(EXPRESSION_PATH, index_col=0)
    print(f"Expression shape: {expr_df.shape} (cell_lines x genes)")
    
    # Transpose: genes as rows, cell lines as columns
    expr_df = expr_df.T
    print(f"Transposed: {expr_df.shape} (genes x cell_lines)")
    
    HAS_EXPRESSION = True
else:
    print("ERROR: Expression data not found!")
    HAS_EXPRESSION = False

In [ ]:
# Find overlapping genes and cell lines
if HAS_EXPRESSION:
    print("\nMatching genes and cell lines...")
    
    # Gene names might have different formats - try to clean them
    def clean_gene_name(name):
        """Extract gene symbol from names like 'GENE (12345)' or 'GENE'."""
        if ' (' in str(name):
            return str(name).split(' (')[0]
        return str(name)
    
    # Clean gene names
    crispr_genes_clean = {clean_gene_name(g): g for g in crispr_df.index}
    expr_genes_clean = {clean_gene_name(g): g for g in expr_df.index}
    
    # Find common genes (by cleaned name)
    common_gene_names = set(crispr_genes_clean.keys()) & set(expr_genes_clean.keys())
    print(f"Common genes (by symbol): {len(common_gene_names)}")
    
    # Map back to original names
    crispr_common_genes = [crispr_genes_clean[g] for g in common_gene_names if g in crispr_genes_clean]
    expr_common_genes = [expr_genes_clean[g] for g in common_gene_names if g in expr_genes_clean]
    
    # Find common cell lines
    common_cells = list(set(crispr_df.columns) & set(expr_df.columns))
    print(f"Common cell lines: {len(common_cells)}")
    
    # Create aligned dataframes
    # We need to align by gene SYMBOL not full name
    crispr_df_aligned = crispr_df.loc[crispr_common_genes, common_cells].copy()
    crispr_df_aligned.index = [clean_gene_name(g) for g in crispr_df_aligned.index]
    
    expr_df_aligned = expr_df.loc[expr_common_genes, common_cells].copy()
    expr_df_aligned.index = [clean_gene_name(g) for g in expr_df_aligned.index]
    
    # Make sure indices match
    common_idx = list(set(crispr_df_aligned.index) & set(expr_df_aligned.index))
    crispr_df_aligned = crispr_df_aligned.loc[common_idx]
    expr_df_aligned = expr_df_aligned.loc[common_idx]
    
    print(f"\nFinal aligned shapes:")
    print(f"  CRISPR: {crispr_df_aligned.shape}")
    print(f"  Expression: {expr_df_aligned.shape}")
    
    # Use aligned data
    crispr_df = crispr_df_aligned
    expr_df = expr_df_aligned

## 2. Data Preprocessing

In [ ]:
# Binarize CRISPR scores
THRESHOLD = -0.5  # Standard DepMap threshold

binary_mat = (crispr_df < THRESHOLD).astype(int)
print(f"Binary matrix: {binary_mat.shape}")
print(f"Essential calls: {binary_mat.sum().sum():,} / {binary_mat.size:,} ({100*binary_mat.sum().sum()/binary_mat.size:.1f}%)")

In [ ]:
# Calculate gene consensus
gene_consensus = binary_mat.mean(axis=1)

# Define thresholds
HIGH_THRESH = 0.9  # Pan-essential
LOW_THRESH = 0.1   # Non-essential

# Classify genes
pan_mask = gene_consensus >= HIGH_THRESH
non_mask = gene_consensus <= LOW_THRESH
context_mask = ~pan_mask & ~non_mask

print(f"Pan-essential (≥{HIGH_THRESH:.0%}): {pan_mask.sum()} genes")
print(f"Non-essential (≤{LOW_THRESH:.0%}): {non_mask.sum()} genes")
print(f"Context-dependent: {context_mask.sum()} genes")

## 3. V2 Predictor with Expression Features

In [ ]:
class HedgeFundPredictorV2:
    """
    V2: Uses expression features for context-dependent genes.
    
    Key insight: Context-dependent genes are essential when EXPRESSED.
    A gene with 50% consensus is essential in cells where it's highly expressed,
    and non-essential where it's lowly expressed.
    """
    
    def __init__(self, high_thresh=0.9, low_thresh=0.1):
        self.high_thresh = high_thresh
        self.low_thresh = low_thresh
        self.ml_model = GradientBoostingClassifier(
            n_estimators=100, 
            max_depth=5,
            learning_rate=0.1,
            subsample=0.8
        )
        self.scaler = StandardScaler()
        self.is_fitted = False
        
    def _extract_features(self, binary_mat, expr_mat, gene, cell_line, other_cells):
        """
        Extract features for a (gene, cell_line) pair.
        """
        # Consensus features
        consensus = binary_mat.loc[gene, other_cells].mean()
        variance = binary_mat.loc[gene, other_cells].var()
        
        # Expression features
        expr_value = expr_mat.loc[gene, cell_line]
        expr_mean = expr_mat.loc[gene, other_cells].mean()
        expr_std = expr_mat.loc[gene, other_cells].std() + 1e-6
        expr_zscore = (expr_value - expr_mean) / expr_std
        
        # Expression percentile
        expr_all = expr_mat.loc[gene, :].values
        expr_percentile = (expr_all < expr_value).mean()
        
        # Interaction features
        consensus_x_expr = consensus * expr_value
        
        return [
            consensus,
            variance,
            expr_value,
            expr_zscore,
            expr_percentile,
            consensus_x_expr,
            consensus ** 2,
            expr_value ** 2
        ]
    
    def fit(self, binary_mat, expr_mat, n_train_cells=100):
        """
        Train the ML model on context-dependent genes.
        """
        print("Training V2 model with expression features...")
        
        # Get context-dependent genes
        gene_consensus = binary_mat.mean(axis=1)
        context_genes = gene_consensus[(gene_consensus > self.low_thresh) & 
                                        (gene_consensus < self.high_thresh)].index.tolist()
        
        print(f"  Context-dependent genes: {len(context_genes)}")
        
        # Sample cell lines for training
        all_cells = list(binary_mat.columns)
        train_cells = all_cells[:n_train_cells]
        
        # Build training data
        X_train = []
        y_train = []
        
        print(f"  Building training data from {len(train_cells)} cell lines...")
        for cell_line in tqdm(train_cells, desc="  Training"):
            other_cells = [c for c in all_cells if c != cell_line]
            
            for gene in context_genes:
                try:
                    features = self._extract_features(binary_mat, expr_mat, gene, cell_line, other_cells)
                    label = binary_mat.loc[gene, cell_line]
                    
                    if not np.isnan(features).any():
                        X_train.append(features)
                        y_train.append(label)
                except Exception as e:
                    continue
        
        X_train = np.array(X_train)
        y_train = np.array(y_train)
        
        print(f"  Training samples: {len(X_train):,}")
        print(f"  Positive rate: {y_train.mean():.1%}")
        
        # Scale and fit
        X_train_scaled = self.scaler.fit_transform(X_train)
        self.ml_model.fit(X_train_scaled, y_train)
        
        # Report training accuracy
        y_pred = self.ml_model.predict(X_train_scaled)
        acc = accuracy_score(y_train, y_pred)
        bal_acc = balanced_accuracy_score(y_train, y_pred)
        
        print(f"  Training accuracy: {acc:.3f}")
        print(f"  Training balanced accuracy: {bal_acc:.3f}")
        
        # Feature importance
        feature_names = ['consensus', 'variance', 'expression', 'expr_zscore', 
                        'expr_percentile', 'cons_x_expr', 'consensus^2', 'expr^2']
        importances = self.ml_model.feature_importances_
        print(f"\n  Feature importances:")
        for name, imp in sorted(zip(feature_names, importances), key=lambda x: -x[1]):
            print(f"    {name:20s}: {imp:.3f}")
        
        self.is_fitted = True
        return self
    
    def predict(self, binary_mat, expr_mat, cell_line):
        """
        Predict essentiality for one cell line.
        """
        all_cells = list(binary_mat.columns)
        other_cells = [c for c in all_cells if c != cell_line]
        
        # Calculate consensus
        consensus = binary_mat[other_cells].mean(axis=1)
        
        # Initialize predictions
        predictions = pd.Series(index=binary_mat.index, dtype=float)
        
        # Pan-essential: predict 1
        pan_mask = consensus >= self.high_thresh
        predictions[pan_mask] = 1
        
        # Non-essential: predict 0
        non_mask = consensus <= self.low_thresh
        predictions[non_mask] = 0
        
        # Context-dependent: use ML
        context_mask = ~pan_mask & ~non_mask
        context_genes = predictions[context_mask].index.tolist()
        
        if len(context_genes) > 0 and self.is_fitted:
            X_test = []
            valid_genes = []
            
            for gene in context_genes:
                try:
                    features = self._extract_features(binary_mat, expr_mat, gene, cell_line, other_cells)
                    if not np.isnan(features).any():
                        X_test.append(features)
                        valid_genes.append(gene)
                except:
                    continue
            
            if len(X_test) > 0:
                X_test = np.array(X_test)
                X_test_scaled = self.scaler.transform(X_test)
                ml_pred = self.ml_model.predict(X_test_scaled)
                
                for gene, pred in zip(valid_genes, ml_pred):
                    predictions[gene] = pred
        
        return predictions.astype(int)

print("HedgeFundPredictorV2 defined!")

## 4. Train and Evaluate

In [ ]:
# Initialize and train predictor
if HAS_EXPRESSION:
    predictor_v2 = HedgeFundPredictorV2()
    predictor_v2.fit(binary_mat, expr_df, n_train_cells=100)
else:
    print("Cannot train V2 without expression data!")

In [ ]:
# Evaluate on held-out cell lines
N_TEST = 50  # Number of cell lines to test

if HAS_EXPRESSION and predictor_v2.is_fitted:
    # Use cell lines NOT in training
    all_cells = list(binary_mat.columns)
    test_cells = all_cells[100:100+N_TEST]  # Skip training cells
    
    all_y_true = []
    all_y_pred = []
    
    print(f"Evaluating on {len(test_cells)} held-out cell lines...")
    for cell_line in tqdm(test_cells):
        y_true = binary_mat[cell_line].values
        y_pred = predictor_v2.predict(binary_mat, expr_df, cell_line).values
        
        all_y_true.extend(y_true)
        all_y_pred.extend(y_pred)
    
    y_true = np.array(all_y_true)
    y_pred = np.array(all_y_pred)
    
    print("\n" + "="*60)
    print("V2 RESULTS (with Expression Features)")
    print("="*60)
    print(f"Accuracy:          {accuracy_score(y_true, y_pred):.3f}")
    print(f"Balanced Accuracy: {balanced_accuracy_score(y_true, y_pred):.3f}")
    print(f"F1 Score:          {f1_score(y_true, y_pred):.3f}")

In [ ]:
# Analyze by gene category
if HAS_EXPRESSION and predictor_v2.is_fitted:
    gene_consensus_full = binary_mat.mean(axis=1)
    
    categories = {
        'Pan-essential (≥90%)': gene_consensus_full >= 0.9,
        'Common (50-90%)': (gene_consensus_full >= 0.5) & (gene_consensus_full < 0.9),
        'Context-dep (10-50%)': (gene_consensus_full >= 0.1) & (gene_consensus_full < 0.5),
        'Rarely ess (0-10%)': (gene_consensus_full > 0) & (gene_consensus_full < 0.1),
        'Non-essential (0%)': gene_consensus_full == 0
    }
    
    print("\nAccuracy by Gene Category:")
    print("-" * 60)
    
    results_v2 = []
    for cat_name, mask in categories.items():
        n_genes = mask.sum()
        if n_genes > 0:
            mask_expanded = np.tile(mask.values, N_TEST)
            
            if mask_expanded.sum() > 0:
                acc = accuracy_score(y_true[mask_expanded], y_pred[mask_expanded])
                results_v2.append({'Category': cat_name, 'Accuracy': acc, 'N_Genes': n_genes})
                print(f"{cat_name:25s}: {acc:.1%} (n={n_genes:,})")

In [ ]:
# Compare V1 vs V2
if HAS_EXPRESSION and predictor_v2.is_fitted:
    # V1 results (from your screenshot)
    v1_results = {
        'Pan-essential (≥90%)': 0.982,
        'Common (50-90%)': 0.434,
        'Context-dep (10-50%)': 0.439,
        'Rarely ess (0-10%)': 0.991,
        'Non-essential (0%)': 1.000
    }
    
    # Plot comparison
    fig, ax = plt.subplots(figsize=(12, 6))
    
    x = np.arange(len(results_v2))
    width = 0.35
    
    v1_accs = [v1_results.get(r['Category'], 0) for r in results_v2]
    v2_accs = [r['Accuracy'] for r in results_v2]
    cats = [r['Category'] for r in results_v2]
    
    bars1 = ax.bar(x - width/2, v1_accs, width, label='V1 (Consensus Only)', color='#ff6b6b', alpha=0.8)
    bars2 = ax.bar(x + width/2, v2_accs, width, label='V2 (+ Expression)', color='#4ecdc4', alpha=0.8)
    
    ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random')
    ax.set_ylabel('Accuracy')
    ax.set_title('V1 vs V2: Prediction Accuracy by Gene Category')
    ax.set_xticks(x)
    ax.set_xticklabels(cats, rotation=45, ha='right')
    ax.legend()
    ax.set_ylim(0, 1.1)
    
    # Add value labels
    for bar, acc in zip(bars1, v1_accs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                f'{acc:.1%}', ha='center', va='bottom', fontsize=9)
    for bar, acc in zip(bars2, v2_accs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                f'{acc:.1%}', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
    # Print improvement
    print("\n" + "="*60)
    print("IMPROVEMENT SUMMARY")
    print("="*60)
    for cat, v1, v2 in zip(cats, v1_accs, v2_accs):
        diff = v2 - v1
        emoji = "✅" if diff > 0.05 else "➡️" if diff > -0.05 else "❌"
        print(f"{cat:25s}: {v1:.1%} → {v2:.1%} ({diff:+.1%}) {emoji}")

## 5. Save Results

In [ ]:
# Save V2 results
if HAS_EXPRESSION:
    OUTPUT_PATH = os.path.join(DATA_DIR, 'gene_essentiality_results_v2.csv')
    
    results_df = pd.DataFrame({
        'gene': binary_mat.index,
        'consensus': gene_consensus.values,
        'n_essential_in': binary_mat.sum(axis=1).values,
        'n_cell_lines': binary_mat.shape[1],
        'mean_expression': expr_df.mean(axis=1).values
    }).sort_values('consensus', ascending=False)
    
    results_df.to_csv(OUTPUT_PATH, index=False)
    print(f"Results saved to: {OUTPUT_PATH}")